<a href="https://colab.research.google.com/github/Nutanpatil06/Advanced_RAG_With_Sentence_Window_Retrieval/blob/main/Advanced_RAG_With_Sentence_Window_Search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install llama-index

In [ ]:
!pip install llama-index-embeddings-huggingface

In [ ]:
!pip install llama-index-vector-stores-qdrant

In [ ]:
!pip install llama-index-llms-gemini

In [ ]:
!pip install -q llama-index google-generativeai

In [ ]:
import os
import sys
import pprint

In [ ]:
from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    load_index_from_storage,
    StorageContext,
    ServiceContext,
    Document
)

In [ ]:
from llama_index.core.node_parser import SentenceWindowNodeParser

In [ ]:
from llama_index.core.text_splitter import SentenceSplitter

In [ ]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

In [ ]:
from llama_index.core.schema import MetadataMode

In [ ]:
from llama_index.core.postprocessor import MetadataReplacementPostProcessor

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Loading embedding model
embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-mpnet-base-v2", max_length=512)

In [ ]:
import os
GOOGLE_API_KEY = "GOOGLE_API_KEY"  #<-- Enter your API key here
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

In [ ]:
import google.generativeai as genai
for m in genai.list_models():
  if "generateContent" in m.supported_generation_methods:
    print(m.name)

In [ ]:
from llama_index.llms.gemini import Gemini
llm = Gemini(model="models/gemini-2.5-flash")

In [ ]:
from llama_index.core.llms import ChatMessage
messages = [
    ChatMessage(role="user", content="Hello Friend!"),
]

response = llm.chat(messages)
print(response)

# Ingest and retrival of the Data

In [ ]:
"""# create local directory and retrieve file from external source
!mkdir -p 'my_data'
!wget 'https://www.gutenberg.org/cache/epub/72306/pg72306.txt' -O './my_data/teahistory.txt'
!wget 'https://www.gutenberg.org/cache/epub/11367.txt' -O './my_data/chinahistory.txt'
"""

In [ ]:
documents = SimpleDirectoryReader(input_files=["/content/my_data/got_book.txt"]).load_data()

In [ ]:
documents

In [ ]:
#inspect the documents
print("length of doc: "+ str(len(documents)))
print("----")
#pprint(documents)

In [ ]:
documents[0].metadata

In [ ]:
# create the sentence window node parser w/ default settings
sentence_node_parser = SentenceWindowNodeParser.from_defaults(
    window_size=3,
    window_metadata_key="window",
    original_text_metadata_key="original_text"
)

In [ ]:
nodes = sentence_node_parser.get_nodes_from_documents(documents)

In [ ]:
base_node_parser = SentenceSplitter()

In [ ]:
base_nodes = base_node_parser.get_nodes_from_documents(documents)

In [ ]:
'''from llama_index.core import Settings
Settings.llm = llm
Settings.embed_model = embed_model
Settings.text_splitter = base_node_parser'''

In [ ]:
len(nodes)

In [ ]:
len(base_nodes)

In [ ]:
print("------")
print("SENTENCE NODES")
print("------")
print(nodes[7])
print("------")
print("BASE NODES")
print("------")
print(base_nodes[7])

In [ ]:
nodes[7].text

In [ ]:
base_nodes[7].text

In [ ]:
dict(nodes[7])

In [ ]:
dict(base_nodes[7])

In [ ]:
from llama_index.core import Settings

Settings.llm = llm
Settings.embed_model = embed_model
Settings.node_parser = sentence_node_parser

# The variable ctx_sentence is no longer directly needed with global settings
#ctx_sentence = ServiceContext.from_defaults(llm=llm, embed_model=embed_model, node_parser=sentence_node_parser)

In [ ]:
from llama_index.core import Settings

Settings.llm = llm
Settings.embed_model = embed_model
Settings.node_parser = base_node_parser

#ctx_base = ServiceContext.from_defaults(llm=llm, embed_model=embed_model, node_parser=base_node_parser)
# The variable ctx_base is no longer directly needed with global settings

In [ ]:
from llama_index.vector_stores.qdrant import QdrantVectorStore

In [ ]:
import qdrant_client

In [ ]:
client = qdrant_client.QdrantClient(
    "QDRANT_END_POINTS",   #<-- Add your Qdrant end point here
    api_key="QDRANT_API_KEY",   #<-- Add your Qdrant API key here
)

In [ ]:
'''client = qdrant_client.QdrantClient(
  # you can use :memory: mode for fast and light-weight experiments,
  # it does not require to have Qdrant deployed anywhere
  # but requires qdrant-client >= 1.1.1
  # location=":memory:"
  # otherwise set Qdrant instance address with:
  # uri="http://<host>:<port>"
  # otherwise set Qdrant instance with host and port:
  host="localhost",
  port=6333
  # set API KEY for Qdrant Cloud
  # api_key="<qdrant-api-key>"
)'''

In [ ]:
vector_store = QdrantVectorStore(client=client, collection_name="got_sent_node")

In [ ]:
storage_context = StorageContext.from_defaults(vector_store=vector_store)

In [ ]:
index = VectorStoreIndex.from_documents(documents, storage_context=storage_context)

In [ ]:
vector_store2 = QdrantVectorStore(client=client, collection_name="got_base_node")

In [ ]:
storage_context2 = StorageContext.from_defaults(vector_store=vector_store2)

In [ ]:
index2 = VectorStoreIndex.from_documents(documents, storage_context=storage_context2)

In [ ]:
#sentence_index = VectorStoreIndex(nodes, service_context=ctx_sentence, service_context=ctx=base)

In [ ]:
#base_index = VectorStoreIndex(base_ndes, service_context=ctx-base)

In [ ]:
'''sentence_index.storage_context.persist(persist_dir="./sentence_index")
base_index.storage_context.persist(persist_dir="./base_index")'''

In [ ]:
#Download to own computer for backup

"""!zip -r ./indexes.zip ./*_index

from google.colab import files
files.download("./indexes.zip")"""

In [ ]:
'''# rebuild storage context
SC_retrieved_sentence = StorageConext.from_defaults(persist_dir="./sentence_index")
SC_retrieved_base = StorageConext.from_defaults(persist_dir="./base_index")'''

In [ ]:
'''# load index
retrieved_sentence_index = load_index-from_storage(SC_retrieved-sentence)
retrieved_base_index = load_index-from-storage(SC_retrieved-base)'''

In [ ]:
'''sentence_query_engine = sentence_index.as_query_engine(
    similarity_top_k=5,
    verbose=True,
    # the target key defaults to `window1 to match the node_parser's default
    node_postprocessors=[
        MetadataReplacementPostProcessor(target_metadata_key="window")
    ],
)'''

In [ ]:
'''base-query-engine = base_index.as_query_engine(
  similarity_top_k=5,
  verbose=True,
)'''

In [ ]:
from llama_index.core.postprocessor import MetadataReplacementPostProcessor

In [ ]:
sentence_query_engine = index.as_query_engine(
    similarity_top_k=3,
    verbose=True,
    # the target key defaults to `window` to match the node_parser's default
    node_postprocessors=[
        MetadataReplacementPostProcessor(target_metadata_key="window")
    ],
)

In [ ]:
base_query_engine = index2.as_query_engine(
  similarity_top_k=3,
  verbose=True,
)

# Generation of the data

In [ ]:
question = "How long have Gared adn Will been part of the Night's Watch?"

In [ ]:
question = "Who is the Ser Waymar Royce?"

In [ ]:
base_response = base_query_engine.query(
    question
)
print(base_response)

In [ ]:
sentense_response = sentence_query_engine.query(
    question
)
print(sentense_response)